<a href="https://colab.research.google.com/github/samar8x/CVIP_Project/blob/main/Notebooks/01_Setup_and_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


startup cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/CV_Project")

CONFIG_DIR = PROJECT_ROOT / "config"
RESULTS_DIR = PROJECT_ROOT / "results"
LOGS_DIR = PROJECT_ROOT / "logs"
DATASETS_DIR = PROJECT_ROOT / "datasets"
MODELS_DIR = PROJECT_ROOT / "models"
NOTEBOOKS_DIR = PROJECT_ROOT / "Notebooks"

print("Project paths ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project paths ready.


Create the permanent project folder

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/CV_Project")

folders = [
    PROJECT_ROOT,
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "datasets" / "archives",
    PROJECT_ROOT / "datasets" / "cityscapes",
    PROJECT_ROOT / "datasets" / "acdc",
    PROJECT_ROOT / "datasets" / "samples",

    PROJECT_ROOT / "models",

    PROJECT_ROOT / "results",
    PROJECT_ROOT / "results" / "cityscapes",
    PROJECT_ROOT / "results" / "acdc",
    PROJECT_ROOT / "results" / "metrics",
    PROJECT_ROOT / "results" / "predictions",
    PROJECT_ROOT / "results" / "plots",
    PROJECT_ROOT / "results" / "visual_examples",

    PROJECT_ROOT / "logs",
    PROJECT_ROOT / "config",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project created successfully:")
print(PROJECT_ROOT)

Project created successfully:
/content/drive/MyDrive/CV_Project


Verify the folders

In [ ]:
for folder in folders:
    status = "✓" if folder.exists() else "✗"
    print(f"{status} {folder}")

✓ /content/drive/MyDrive/CV_Project
✓ /content/drive/MyDrive/CV_Project/datasets
✓ /content/drive/MyDrive/CV_Project/datasets/archives
✓ /content/drive/MyDrive/CV_Project/datasets/cityscapes
✓ /content/drive/MyDrive/CV_Project/datasets/acdc
✓ /content/drive/MyDrive/CV_Project/datasets/samples
✓ /content/drive/MyDrive/CV_Project/models
✓ /content/drive/MyDrive/CV_Project/results
✓ /content/drive/MyDrive/CV_Project/results/cityscapes
✓ /content/drive/MyDrive/CV_Project/results/acdc
✓ /content/drive/MyDrive/CV_Project/results/metrics
✓ /content/drive/MyDrive/CV_Project/results/predictions
✓ /content/drive/MyDrive/CV_Project/results/plots
✓ /content/drive/MyDrive/CV_Project/results/visual_examples
✓ /content/drive/MyDrive/CV_Project/logs
✓ /content/drive/MyDrive/CV_Project/config


Check the Colab hardware

In [ ]:
import torch
import platform
import sys

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("No GPU currently enabled.")

print("Platform:", platform.platform())

Python version: 3.13.16
PyTorch version: 2.11.0+cpu
CUDA available: False
No GPU currently enabled.
Platform: Linux-6.6.159+-x86_64-with-glibc2.39


Save the environment information permanently

In [ ]:
import json
import datetime
import torch
import sys
import platform

environment_info = {
    "created": datetime.datetime.now().isoformat(),
    "python_version": sys.version,
    "pytorch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0)
           if torch.cuda.is_available()
           else None,
    "platform": platform.platform(),
}

env_path = PROJECT_ROOT / "config" / "environment.json"

with open(env_path, "w") as f:
    json.dump(environment_info, f, indent=4)

print("Environment information saved to:")
print(env_path)

Environment information saved to:
/content/drive/MyDrive/CV_Project/config/environment.json


Create the permanent experiment configuration

In [ ]:
project_config = {
    "project_title":
        "Per-Class Analysis of Semantic Segmentation Degradation "
        "Under Adverse Urban Driving Conditions",

    "datasets": {
        "clear": "Cityscapes",
        "adverse": "ACDC"
    },

    "conditions": [
        "clear",
        "fog",
        "rain",
        "snow",
        "night"
    ],

    "model": {
        "architecture": "SegFormer-B2",
        "training_domain": "Cityscapes",
        "fine_tuning": False,
        "test_time_adaptation": False
    },

    "matched_analysis": {
        "primary_split": "validation",
        "primary_pairs": 203,
        "expanded_robustness_split": "train+validation",
        "expanded_pairs": 1003
    },

    "acdc_invalid_mask": {
        "mask_type": "invIds",
        "exclude_invalid_pixels": True
    },

    "main_metric": "per-class IoU",

    "degradation_formula":
        "Clear IoU - Adverse Condition IoU",

    "num_classes": 19,

    "classes": [
        "road",
        "sidewalk",
        "building",
        "wall",
        "fence",
        "pole",
        "traffic light",
        "traffic sign",
        "vegetation",
        "terrain",
        "sky",
        "person",
        "rider",
        "car",
        "truck",
        "bus",
        "train",
        "motorcycle",
        "bicycle"
    ]
}

config_path = PROJECT_ROOT / "config" / "project_config.json"

with open(config_path, "w") as f:
    json.dump(project_config, f, indent=4)

print("Configuration saved:")
print(config_path)

Configuration saved:
/content/drive/MyDrive/CV_Project/config/project_config.json


Create a progress tracker

In [ ]:
progress = {
    "datasets": {
        "cityscapes_downloaded": False,
        "cityscapes_prepared": False,
        "acdc_downloaded": False,
        "acdc_prepared": False
    },

    "model": {
        "segformer_loaded": False
    },

    "evaluation": {
        "cityscapes_complete": False,
        "fog_complete": False,
        "rain_complete": False,
        "snow_complete": False,
        "night_complete": False
    },

    "analysis": {
        "iou_complete": False,
        "degradation_complete": False,
        "plots_complete": False,
        "visual_examples_complete": False
    }
}

progress_path = PROJECT_ROOT / "config" / "progress.json"

if not progress_path.exists():
    with open(progress_path, "w") as f:
        json.dump(progress, f, indent=4)

    print("New progress tracker created.")
else:
    print("Progress tracker already exists — keeping previous progress.")

print(progress_path)

Progress tracker already exists — keeping previous progress.
/content/drive/MyDrive/CV_Project/config/progress.json


Create a helper to update progress

In [ ]:
def update_progress(section, item, value=True):
    with open(progress_path, "r") as f:
        current_progress = json.load(f)

    current_progress[section][item] = value

    with open(progress_path, "w") as f:
        json.dump(current_progress, f, indent=4)

    print(f"Updated: {section} -> {item} = {value}")

Create a permanent run log

In [ ]:
from datetime import datetime

LOG_FILE = PROJECT_ROOT / "logs" / "experiment_log.txt"

def log_message(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    line = f"[{timestamp}] {message}\n"

    with open(LOG_FILE, "a") as f:
        f.write(line)

    print(line.strip())

log_message("CV project environment initialized.")

[2026-10-27 16:50:10] CV project environment initialized.


Check available local Colab disk

In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /


## Dataset Preparation

This section verifies the dataset archives and prepares the data required for both the primary validation analysis and the expanded matched robustness analysis.

The project uses:

- **Cityscapes validation images** as the clear-condition source-domain reference.
- **ACDC validation images** for fog, rain, snow, and night.
- **ACDC `val_ref` images** as the primary matched normal-condition references.
- **ACDC `train` and `train_ref` images** for the expanded matched robustness analysis.
- Semantic-segmentation ground-truth masks from Cityscapes and ACDC.
- ACDC `invIds` masks so invalid or uncertain adverse-condition pixels can be excluded from IoU computation.

The primary matched evaluation uses the ACDC validation correspondences. The expanded robustness analysis uses the available labeled train and validation correspondences to improve support for rare semantic classes.

In [ ]:
from pathlib import Path

# ------------------------------------------------------------
# Main project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/CV_Project"
)

DATASET_DIR = PROJECT_ROOT / "datasets"

ARCHIVE_DIR = DATASET_DIR / "archives"

ACDC_DIR = DATASET_DIR / "acdc"

CITYSCAPES_DIR = DATASET_DIR / "cityscapes"


# ------------------------------------------------------------
# Dataset archive paths
# ------------------------------------------------------------

CITYSCAPES_RGB_ZIP = (
    CITYSCAPES_DIR
    / "leftImg8bit.zip"
)

CITYSCAPES_GT_ZIP = (
    ARCHIVE_DIR
    / "gtFine_trainvaltest.zip"
)

ACDC_RGB_ZIP = (
    ACDC_DIR
    / "rgb_anon.zip"
)

ACDC_GT_ZIP = (
    ARCHIVE_DIR
    / "gt_trainval.zip"
)

ACDC_GT_REF_ZIP = (
    ARCHIVE_DIR
    / "gt_trainval_ref.zip"
)


# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

dataset_archives = {
    "Cityscapes RGB": CITYSCAPES_RGB_ZIP,
    "Cityscapes GT": CITYSCAPES_GT_ZIP,
    "ACDC RGB": ACDC_RGB_ZIP,
    "ACDC GT": ACDC_GT_ZIP,
    "ACDC Reference GT": ACDC_GT_REF_ZIP,
}

for name, path in dataset_archives.items():

    print(
        f"{name}:",
        "FOUND" if path.exists() else "MISSING"
    )

    print(" ", path)

Cityscapes RGB: FOUND
  /content/drive/MyDrive/CV_Project/datasets/cityscapes/leftImg8bit.zip
Cityscapes GT: FOUND
  /content/drive/MyDrive/CV_Project/datasets/archives/gtFine_trainvaltest.zip
ACDC RGB: FOUND
  /content/drive/MyDrive/CV_Project/datasets/acdc/rgb_anon.zip
ACDC GT: FOUND
  /content/drive/MyDrive/CV_Project/datasets/archives/gt_trainval.zip
ACDC Reference GT: FOUND
  /content/drive/MyDrive/CV_Project/datasets/archives/gt_trainval_ref.zip


verify ZIP integrity

In [ ]:
import zipfile

for name, path in dataset_archives.items():

    print(f"Checking {name}...")

    try:

        with zipfile.ZipFile(path, "r") as z:
            bad_file = z.testzip()

        if bad_file is None:
            print("   ✓ ZIP is valid")
        else:
            print(
                "   ✗ Corrupted member:",
                bad_file
            )

    except zipfile.BadZipFile:
        print("   ✗ Invalid ZIP file")

Checking Cityscapes RGB...
   ✓ ZIP is valid
Checking Cityscapes GT...
   ✓ ZIP is valid
Checking ACDC RGB...
   ✓ ZIP is valid
Checking ACDC GT...
   ✓ ZIP is valid
Checking ACDC Reference GT...
   ✓ ZIP is valid


Important storage decision

In [ ]:
LOCAL_DATA_ROOT = Path("/content/cv_data")

CITYSCAPES_LOCAL = LOCAL_DATA_ROOT / "cityscapes"
ACDC_LOCAL = LOCAL_DATA_ROOT / "acdc"

CITYSCAPES_LOCAL.mkdir(parents=True, exist_ok=True)
ACDC_LOCAL.mkdir(parents=True, exist_ok=True)

print("Local temporary dataset directory:")
print(LOCAL_DATA_ROOT)

Local temporary dataset directory:
/content/cv_data


Extract only Cityscapes validation images

In [ ]:
import zipfile

CITYSCAPES_RGB_OUT = CITYSCAPES_LOCAL / "rgb"

CITYSCAPES_RGB_OUT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(CITYSCAPES_RGB_ZIP, "r") as z:
    selected = [
        name
        for name in z.namelist()
        if name.startswith("leftImg8bit/val/")
        and name.endswith("_leftImg8bit.png")
    ]

    print("Cityscapes validation images selected:", len(selected))

    for i, name in enumerate(selected, start=1):
        z.extract(name, CITYSCAPES_RGB_OUT)

        if i % 100 == 0 or i == len(selected):
            print(f"Extracted {i}/{len(selected)}")

Cityscapes validation images selected: 500
Extracted 100/500
Extracted 200/500
Extracted 300/500
Extracted 400/500
Extracted 500/500


Verify Cityscapes image count

In [ ]:
cityscapes_rgb_files = sorted(
    CITYSCAPES_RGB_OUT.rglob("*_leftImg8bit.png")
)

print("Cityscapes RGB validation images:",
      len(cityscapes_rgb_files))

Cityscapes RGB validation images: 500


Extract ACDC train/validation images and their normal-condition references

In [ ]:
ACDC_RGB_OUT = ACDC_LOCAL / "rgb"

ACDC_RGB_OUT.mkdir(parents=True, exist_ok=True)

conditions = ["fog", "rain", "snow", "night"]

# Primary validation analysis + expanded matched robustness analysis
wanted_splits = [
    "train",
    "train_ref",
    "val",
    "val_ref"
]

with zipfile.ZipFile(ACDC_RGB_ZIP, "r") as z:

    selected = []

    for name in z.namelist():

        if "__MACOSX" in name:
            continue

        if not name.endswith(".png"):
            continue

        parts = name.split("/")

        # Expected:
        # rgb_anon/<condition>/<split>/<sequence>/<image>.png
        if len(parts) < 5:
            continue

        root = parts[0]
        condition = parts[1]
        split = parts[2]

        if (
            root == "rgb_anon"
            and condition in conditions
            and split in wanted_splits
        ):
            selected.append(name)

    print(
        "ACDC train/validation/reference images selected:",
        len(selected)
    )

    for i, name in enumerate(selected, start=1):

        z.extract(
            name,
            ACDC_RGB_OUT
        )

        if i % 500 == 0 or i == len(selected):

            print(
                f"Extracted {i}/{len(selected)}"
            )

ACDC train/validation/reference images selected: 4012
Extracted 500/4012
Extracted 1000/4012
Extracted 1500/4012
Extracted 2000/4012
Extracted 2500/4012
Extracted 3000/4012
Extracted 3500/4012
Extracted 4000/4012
Extracted 4012/4012


### Verify full ACDC RGB split counts

The full RGB archive should contain `train`, `train_ref`, `val`, and `val_ref` images for all four adverse conditions.

In [ ]:
from collections import Counter

acdc_rgb_files = list(
    (ACDC_RGB_OUT / "rgb_anon").rglob("*.png")
)

rgb_counts = Counter()

for file in acdc_rgb_files:
    parts = file.parts

    condition = None
    split = None

    for c in ["fog", "rain", "snow", "night"]:
        if c in parts:
            condition = c
            break

    for s in ["train", "train_ref", "val", "val_ref"]:
        if s in parts:
            split = s
            break

    if condition is not None and split is not None:
        rgb_counts[(condition, split)] += 1


print("Total extracted ACDC RGB images:", len(acdc_rgb_files))

for condition in conditions:
    print(f"\n{condition.upper()}")

    for split in ["train", "train_ref", "val", "val_ref"]:
        print(
            f"{split:10s}:",
            rgb_counts[(condition, split)]
        )

Total extracted ACDC RGB images: 4012

FOG
train     : 400
train_ref : 400
val       : 100
val_ref   : 100

RAIN
train     : 400
train_ref : 400
val       : 100
val_ref   : 100

SNOW
train     : 400
train_ref : 400
val       : 100
val_ref   : 100

NIGHT
train     : 400
train_ref : 400
val       : 106
val_ref   : 106


Verify ACDC counts

In [ ]:
from collections import Counter

acdc_counts = Counter()

for file in ACDC_RGB_OUT.rglob("*.png"):

    parts = file.relative_to(ACDC_RGB_OUT).parts

    # rgb_anon / condition / split / sequence / file
    if len(parts) >= 5:
        condition = parts[1]
        split = parts[2]

        acdc_counts[(condition, split)] += 1

for condition in conditions:
    for split in wanted_splits:
        print(
            f"{condition:5} {split:7}: "
            f"{acdc_counts[(condition, split)]}"
        )

fog   train  : 400
fog   train_ref: 400
fog   val    : 100
fog   val_ref: 100
rain  train  : 400
rain  train_ref: 400
rain  val    : 100
rain  val_ref: 100
snow  train  : 400
snow  train_ref: 400
snow  val    : 100
snow  val_ref: 100
night train  : 400
night train_ref: 400
night val    : 106
night val_ref: 106


inspect the ground-truth archives

In [ ]:
gt_archives = {
    "Cityscapes GT": CITYSCAPES_GT_ZIP,
    "ACDC GT": ACDC_GT_ZIP,
    "ACDC GT Ref": ACDC_GT_REF_ZIP,
}

for name, path in gt_archives.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    with zipfile.ZipFile(path, "r") as z:
        entries = z.namelist()

    print("Total entries:", len(entries))
    print("\nFirst 20 entries:")

    for entry in entries[:20]:
        print(entry)


Cityscapes GT
Total entries: 40068

First 20 entries:
gtFine_trainvaltest/
__MACOSX/._gtFine_trainvaltest
gtFine_trainvaltest/README
__MACOSX/gtFine_trainvaltest/._README
gtFine_trainvaltest/license.txt
__MACOSX/gtFine_trainvaltest/._license.txt
gtFine_trainvaltest/gtFine/
__MACOSX/gtFine_trainvaltest/._gtFine
gtFine_trainvaltest/gtFine/test/
__MACOSX/gtFine_trainvaltest/gtFine/._test
gtFine_trainvaltest/gtFine/train/
__MACOSX/gtFine_trainvaltest/gtFine/._train
gtFine_trainvaltest/gtFine/val/
__MACOSX/gtFine_trainvaltest/gtFine/._val
gtFine_trainvaltest/gtFine/test/bielefeld/
__MACOSX/gtFine_trainvaltest/gtFine/test/._bielefeld
gtFine_trainvaltest/gtFine/test/leverkusen/
__MACOSX/gtFine_trainvaltest/gtFine/test/._leverkusen
gtFine_trainvaltest/gtFine/test/bonn/
__MACOSX/gtFine_trainvaltest/gtFine/test/._bonn

ACDC GT
Total entries: 10032

First 20 entries:
License.pdf
README.md
gt/fog/train/GOPR0475/GOPR0475_frame_000041_gt_labelIds.png
gt/fog/train/GOPR0475/GOPR0475_frame_000049_gt_l

which mask types are actually present in each ZIP.

In [ ]:
from collections import Counter
import zipfile

def summarize_png_suffixes(zip_path, dataset_name):
    with zipfile.ZipFile(zip_path, "r") as z:
        files = [
            name for name in z.namelist()
            if name.endswith(".png")
        ]

    suffix_counts = Counter()

    for name in files:
        filename = name.split("/")[-1]

        # Keep the useful end of the filename
        if "_gtFine_" in filename:
            suffix = filename.split("_gtFine_")[-1]

        elif "_gt_ref_" in filename:
            suffix = filename.split("_gt_ref_")[-1]

        elif "_gt_" in filename:
            suffix = filename.split("_gt_")[-1]

        else:
            suffix = filename

        suffix_counts[suffix] += 1

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)

    for suffix, count in sorted(suffix_counts.items()):
        print(f"{suffix:30} : {count}")


summarize_png_suffixes(
    CITYSCAPES_GT_ZIP,
    "Cityscapes GT"
)

summarize_png_suffixes(
    ACDC_GT_ZIP,
    "ACDC GT"
)

summarize_png_suffixes(
    ACDC_GT_REF_ZIP,
    "ACDC GT Reference"
)


Cityscapes GT
color.png                      : 10000
instanceIds.png                : 10000
labelIds.png                   : 10000

ACDC GT
invGray.png                    : 2006
invIds.png                     : 2006
labelColor.png                 : 2006
labelIds.png                   : 2006
labelTrainIds.png              : 2006

ACDC GT Reference
labelColor.png                 : 1003
labelIds.png                   : 1003
labelTrainIds.png              : 1003


inspect the train and validation masks:

In [ ]:
def show_split_examples(zip_path, dataset_name, patterns):
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

    matching = []

    for name in names:
        if "__MACOSX" in name:
            continue

        if any(pattern in name for pattern in patterns):
            if name.endswith(".png"):
                matching.append(name)

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)
    print("Matching PNG files:", len(matching))

    print("\nFirst 10:")
    for name in matching[:10]:
        print(name)


show_split_examples(
    CITYSCAPES_GT_ZIP,
    "Cityscapes validation GT",
    ["/gtFine/val/"]
)

show_split_examples(
    ACDC_GT_ZIP,
    "ACDC train + validation GT",
    ["/train/", "/val/"]
)

show_split_examples(
    ACDC_GT_REF_ZIP,
    "ACDC train_ref + val_ref GT",
    ["/train_ref/", "/val_ref/"]
)


Cityscapes validation GT
Matching PNG files: 1500

First 10:
gtFine_trainvaltest/gtFine/val/munster/munster_000046_000019_gtFine_instanceIds.png
gtFine_trainvaltest/gtFine/val/munster/munster_000168_000019_gtFine_instanceIds.png
gtFine_trainvaltest/gtFine/val/munster/munster_000096_000019_gtFine_color.png
gtFine_trainvaltest/gtFine/val/munster/munster_000097_000019_gtFine_instanceIds.png
gtFine_trainvaltest/gtFine/val/munster/munster_000003_000019_gtFine_instanceIds.png
gtFine_trainvaltest/gtFine/val/munster/munster_000121_000019_gtFine_color.png
gtFine_trainvaltest/gtFine/val/munster/munster_000089_000019_gtFine_instanceIds.png
gtFine_trainvaltest/gtFine/val/munster/munster_000107_000019_gtFine_labelIds.png
gtFine_trainvaltest/gtFine/val/munster/munster_000133_000019_gtFine_instanceIds.png
gtFine_trainvaltest/gtFine/val/munster/munster_000071_000019_gtFine_labelIds.png

ACDC train + validation GT
Matching PNG files: 10030

First 10:
gt/fog/train/GOPR0475/GOPR0475_frame_000041_gt_labe

define Cityscapes ID mapping

In [ ]:
# Cityscapes official labelId -> trainId mapping
# Only the 19 evaluation classes are mapped to 0-18.
# Everything else becomes 255 = ignore.

CITYSCAPES_ID_TO_TRAINID = {
    7: 0,    # road
    8: 1,    # sidewalk
    11: 2,   # building
    12: 3,   # wall
    13: 4,   # fence
    17: 5,   # pole
    19: 6,   # traffic light
    20: 7,   # traffic sign
    21: 8,   # vegetation
    22: 9,   # terrain
    23: 10,  # sky
    24: 11,  # person
    25: 12,  # rider
    26: 13,  # car
    27: 14,  # truck
    28: 15,  # bus
    31: 16,  # train
    32: 17,  # motorcycle
    33: 18,  # bicycle
}

IGNORE_INDEX = 255

print("Cityscapes mapping ready.")

Cityscapes mapping ready.


inspect the useful semantic and invalid-mask files

In [ ]:
def count_useful_gt_files(zip_path, dataset_name, rules):
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

    useful = []

    for name in names:
        if "__MACOSX" in name:
            continue

        if not name.endswith(".png"):
            continue

        if all(rule in name for rule in rules):
            useful.append(name)

    print(f"{dataset_name}: {len(useful)} useful files")

    for name in useful[:5]:
        print(" ", name)

    return useful


# Cityscapes
cityscapes_val_labelids = count_useful_gt_files(
    CITYSCAPES_GT_ZIP,
    "Cityscapes val labelIds",
    ["/gtFine/val/", "_gtFine_labelIds.png"]
)


# ACDC semantic labelTrainIds
acdc_train_trainids = count_useful_gt_files(
    ACDC_GT_ZIP,
    "ACDC train labelTrainIds",
    ["/train/", "_gt_labelTrainIds.png"]
)

acdc_val_trainids = count_useful_gt_files(
    ACDC_GT_ZIP,
    "ACDC val labelTrainIds",
    ["/val/", "_gt_labelTrainIds.png"]
)


# ACDC reference semantic GT
acdc_trainref_trainids = count_useful_gt_files(
    ACDC_GT_REF_ZIP,
    "ACDC train_ref labelTrainIds",
    ["/train_ref/", "_gt_ref_labelTrainIds.png"]
)

acdc_valref_trainids = count_useful_gt_files(
    ACDC_GT_REF_ZIP,
    "ACDC val_ref labelTrainIds",
    ["/val_ref/", "_gt_ref_labelTrainIds.png"]
)


# ACDC invalid / uncertainty masks
acdc_train_invalid = count_useful_gt_files(
    ACDC_GT_ZIP,
    "ACDC train invIds",
    ["/train/", "_gt_invIds.png"]
)

acdc_val_invalid = count_useful_gt_files(
    ACDC_GT_ZIP,
    "ACDC val invIds",
    ["/val/", "_gt_invIds.png"]
)

Cityscapes val labelIds: 500 useful files
  gtFine_trainvaltest/gtFine/val/munster/munster_000107_000019_gtFine_labelIds.png
  gtFine_trainvaltest/gtFine/val/munster/munster_000071_000019_gtFine_labelIds.png
  gtFine_trainvaltest/gtFine/val/munster/munster_000044_000019_gtFine_labelIds.png
  gtFine_trainvaltest/gtFine/val/munster/munster_000086_000019_gtFine_labelIds.png
  gtFine_trainvaltest/gtFine/val/munster/munster_000132_000019_gtFine_labelIds.png
ACDC train labelTrainIds: 1600 useful files
  gt/fog/train/GOPR0475/GOPR0475_frame_000041_gt_labelTrainIds.png
  gt/fog/train/GOPR0475/GOPR0475_frame_000049_gt_labelTrainIds.png
  gt/fog/train/GOPR0475/GOPR0475_frame_000096_gt_labelTrainIds.png
  gt/fog/train/GOPR0475/GOPR0475_frame_000116_gt_labelTrainIds.png
  gt/fog/train/GOPR0475/GOPR0475_frame_000122_gt_labelTrainIds.png
ACDC val labelTrainIds: 406 useful files
  gt/fog/val/GOPR0476/GOPR0476_frame_000761_gt_labelTrainIds.png
  gt/fog/val/GOPR0476/GOPR0476_frame_000769_gt_labelTrainI

extract the semantic GT masks and ACDC invalid masks we need

In [ ]:
CITYSCAPES_GT_OUT = Path("/content/cv_data/cityscapes_gt")
ACDC_GT_OUT = Path("/content/cv_data/acdc_gt")
ACDC_GT_REF_OUT = Path("/content/cv_data/acdc_gt_ref")

CITYSCAPES_GT_OUT.mkdir(parents=True, exist_ok=True)
ACDC_GT_OUT.mkdir(parents=True, exist_ok=True)
ACDC_GT_REF_OUT.mkdir(parents=True, exist_ok=True)


def extract_selected(zip_path, selected_files, output_dir, label):
    with zipfile.ZipFile(zip_path, "r") as z:
        for i, name in enumerate(selected_files, start=1):
            z.extract(name, output_dir)

            if i % 200 == 0 or i == len(selected_files):
                print(f"{label}: {i}/{len(selected_files)}")

    print(f"{label} extraction complete.")


# Cityscapes
extract_selected(
    CITYSCAPES_GT_ZIP,
    cityscapes_val_labelids,
    CITYSCAPES_GT_OUT,
    "Cityscapes GT"
)


# ACDC adverse semantic GT
extract_selected(
    ACDC_GT_ZIP,
    acdc_train_trainids,
    ACDC_GT_OUT,
    "ACDC Train GT"
)

extract_selected(
    ACDC_GT_ZIP,
    acdc_val_trainids,
    ACDC_GT_OUT,
    "ACDC Validation GT"
)


# ACDC invalid / uncertainty masks
extract_selected(
    ACDC_GT_ZIP,
    acdc_train_invalid,
    ACDC_GT_OUT,
    "ACDC Train invIds"
)

extract_selected(
    ACDC_GT_ZIP,
    acdc_val_invalid,
    ACDC_GT_OUT,
    "ACDC Validation invIds"
)


# ACDC normal-reference GT
extract_selected(
    ACDC_GT_REF_ZIP,
    acdc_trainref_trainids,
    ACDC_GT_REF_OUT,
    "ACDC Train Ref GT"
)

extract_selected(
    ACDC_GT_REF_ZIP,
    acdc_valref_trainids,
    ACDC_GT_REF_OUT,
    "ACDC Validation Ref GT"
)

Cityscapes GT: 200/500
Cityscapes GT: 400/500
Cityscapes GT: 500/500
Cityscapes GT extraction complete.
ACDC Train GT: 200/1600
ACDC Train GT: 400/1600
ACDC Train GT: 600/1600
ACDC Train GT: 800/1600
ACDC Train GT: 1000/1600
ACDC Train GT: 1200/1600
ACDC Train GT: 1400/1600
ACDC Train GT: 1600/1600
ACDC Train GT extraction complete.
ACDC Validation GT: 200/406
ACDC Validation GT: 400/406
ACDC Validation GT: 406/406
ACDC Validation GT extraction complete.
ACDC Train invIds: 200/1600
ACDC Train invIds: 400/1600
ACDC Train invIds: 600/1600
ACDC Train invIds: 800/1600
ACDC Train invIds: 1000/1600
ACDC Train invIds: 1200/1600
ACDC Train invIds: 1400/1600
ACDC Train invIds: 1600/1600
ACDC Train invIds extraction complete.
ACDC Validation invIds: 200/406
ACDC Validation invIds: 400/406
ACDC Validation invIds: 406/406
ACDC Validation invIds extraction complete.
ACDC Train Ref GT: 200/800
ACDC Train Ref GT: 400/800
ACDC Train Ref GT: 600/800
ACDC Train Ref GT: 800/800
ACDC Train Ref GT extracti

verify extracted semantic and invalid-mask counts

In [ ]:
city_gt_files = list(
    CITYSCAPES_GT_OUT.rglob("*_gtFine_labelIds.png")
)

acdc_gt_files = list(
    ACDC_GT_OUT.rglob("*_gt_labelTrainIds.png")
)

acdc_invalid_files = list(
    ACDC_GT_OUT.rglob("*_gt_invIds.png")
)

acdc_ref_gt_files = list(
    ACDC_GT_REF_OUT.rglob("*_gt_ref_labelTrainIds.png")
)

print("Cityscapes GT:", len(city_gt_files))
print("ACDC semantic GT:", len(acdc_gt_files))
print("ACDC invIds masks:", len(acdc_invalid_files))
print("ACDC Ref GT:", len(acdc_ref_gt_files))


from collections import Counter

def count_by_split_and_condition(files):
    counts = Counter()

    for file in files:
        parts = file.parts

        condition = None
        split = None

        for c in [
            "fog",
            "rain",
            "snow",
            "night"
        ]:
            if c in parts:
                condition = c
                break

        for s in [
            "train",
            "train_ref",
            "val",
            "val_ref"
        ]:
            if s in parts:
                split = s
                break

        if (
            condition is not None
            and split is not None
        ):
            counts[(condition, split)] += 1

    return counts


print("\nACDC semantic GT counts")

semantic_counts = count_by_split_and_condition(
    acdc_gt_files
)

for condition in conditions:
    print(
        condition,
        "train:",
        semantic_counts[(condition, "train")],
        "| val:",
        semantic_counts[(condition, "val")]
    )


print("\nACDC invalid-mask counts")

invalid_counts = count_by_split_and_condition(
    acdc_invalid_files
)

for condition in conditions:
    print(
        condition,
        "train:",
        invalid_counts[(condition, "train")],
        "| val:",
        invalid_counts[(condition, "val")]
    )


print("\nACDC reference GT counts")

ref_counts = count_by_split_and_condition(
    acdc_ref_gt_files
)

for condition in conditions:
    print(
        condition,
        "train_ref:",
        ref_counts[(condition, "train_ref")],
        "| val_ref:",
        ref_counts[(condition, "val_ref")]
    )

Cityscapes GT: 500
ACDC semantic GT: 2006
ACDC invIds masks: 2006
ACDC Ref GT: 1003

ACDC semantic GT counts
fog train: 400 | val: 100
rain train: 400 | val: 100
snow train: 400 | val: 100
night train: 400 | val: 106

ACDC invalid-mask counts
fog train: 400 | val: 100
rain train: 400 | val: 100
snow train: 400 | val: 100
night train: 400 | val: 106

ACDC reference GT counts
fog train_ref: 200 | val_ref: 50
rain train_ref: 200 | val_ref: 50
snow train_ref: 200 | val_ref: 50
night train_ref: 200 | val_ref: 53


save the Cityscapes conversion helper

In [ ]:
import numpy as np

def cityscapes_labelids_to_trainids(label_array):
    train_mask = np.full(
        label_array.shape,
        IGNORE_INDEX,
        dtype=np.uint8
    )

    for label_id, train_id in CITYSCAPES_ID_TO_TRAINID.items():
        train_mask[label_array == label_id] = train_id

    return train_mask

## Final Dataset Verification

This section verifies that the prepared RGB images, semantic-segmentation masks, reference masks, and ACDC invalid/uncertainty masks have the expected counts before model evaluation begins.

The primary matched analysis uses the validation correspondences. The expanded robustness analysis uses the available train and validation correspondences.

In [ ]:
dataset_summary = {
    "Cityscapes": {
        "validation_rgb_images": len(cityscapes_rgb_files),
        "validation_ground_truth_masks": len(city_gt_files),
    },

    "ACDC": {
        "rgb_images_total": len(acdc_rgb_files),

        "fog_train_rgb": rgb_counts[("fog", "train")],
        "fog_train_ref_rgb": rgb_counts[("fog", "train_ref")],
        "fog_val_rgb": rgb_counts[("fog", "val")],
        "fog_val_ref_rgb": rgb_counts[("fog", "val_ref")],

        "rain_train_rgb": rgb_counts[("rain", "train")],
        "rain_train_ref_rgb": rgb_counts[("rain", "train_ref")],
        "rain_val_rgb": rgb_counts[("rain", "val")],
        "rain_val_ref_rgb": rgb_counts[("rain", "val_ref")],

        "snow_train_rgb": rgb_counts[("snow", "train")],
        "snow_train_ref_rgb": rgb_counts[("snow", "train_ref")],
        "snow_val_rgb": rgb_counts[("snow", "val")],
        "snow_val_ref_rgb": rgb_counts[("snow", "val_ref")],

        "night_train_rgb": rgb_counts[("night", "train")],
        "night_train_ref_rgb": rgb_counts[("night", "train_ref")],
        "night_val_rgb": rgb_counts[("night", "val")],
        "night_val_ref_rgb": rgb_counts[("night", "val_ref")],

        "adverse_ground_truth_masks": len(acdc_gt_files),
        "invalid_uncertainty_masks": len(acdc_invalid_files),
        "reference_ground_truth_masks": len(acdc_ref_gt_files),

        "primary_matched_pairs": 203,
        "expanded_matched_pairs": 1003
    }
}

print(json.dumps(dataset_summary, indent=4))

{
    "Cityscapes": {
        "validation_rgb_images": 500,
        "validation_ground_truth_masks": 500
    },
    "ACDC": {
        "rgb_images_total": 4012,
        "fog_train_rgb": 400,
        "fog_train_ref_rgb": 400,
        "fog_val_rgb": 100,
        "fog_val_ref_rgb": 100,
        "rain_train_rgb": 400,
        "rain_train_ref_rgb": 400,
        "rain_val_rgb": 100,
        "rain_val_ref_rgb": 100,
        "snow_train_rgb": 400,
        "snow_train_ref_rgb": 400,
        "snow_val_rgb": 100,
        "snow_val_ref_rgb": 100,
        "night_train_rgb": 400,
        "night_train_ref_rgb": 400,
        "night_val_rgb": 106,
        "night_val_ref_rgb": 106,
        "adverse_ground_truth_masks": 2006,
        "invalid_uncertainty_masks": 2006,
        "reference_ground_truth_masks": 1003,
        "primary_matched_pairs": 203,
        "expanded_matched_pairs": 1003
    }
}


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/CV_Project")

CONFIG_DIR = PROJECT_ROOT / "config"
RESULTS_DIR = PROJECT_ROOT / "results"
LOGS_DIR = PROJECT_ROOT / "logs"
DATASETS_DIR = PROJECT_ROOT / "datasets"

print("CONFIG_DIR:", CONFIG_DIR)

summary_path = CONFIG_DIR / "dataset_summary.json"

with open(summary_path, "w") as f:
    json.dump(dataset_summary, f, indent=4)

print("Dataset summary saved to:")
print(summary_path)

CONFIG_DIR: /content/drive/MyDrive/CV_Project/config
Dataset summary saved to:
/content/drive/MyDrive/CV_Project/config/dataset_summary.json
